# Market Performance Analysis

This notebook analyzes controller performance relative to overall market activity.

**Purpose:**
- Fetch market data (OHLC, volume, trades) for each trading pair
- Calculate controller market share per trading pair
- Generate HTML report grouped by trading pair with market context

**Prerequisites:**
- Run `consolidate_data.ipynb` first to generate consolidated trades

**Output:**
- `market_analysis_report_latest.html` - Market performance report by trading pair

In [1]:
import asyncio
import sys
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path
sys.path.append('/Users/tomasgaudino/PycharmProjects/quants-lab')

from research_notebooks.brigado_v2.data_consolidator import DataConsolidator
from research_notebooks.brigado_v2.file_manager import FileManager
from core.data_sources.clob import CLOBDataSource

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✓ Imports loaded")

✓ Imports loaded


In [2]:
# ============================================================================
#                        LOAD CONSOLIDATED DATA
# ============================================================================

print("\n" + "="*80)
print("LOADING CONSOLIDATED DATA")
print("="*80 + "\n")

consolidator = DataConsolidator()
file_manager = FileManager()
data = consolidator.load_consolidated_data(use_latest=True)

print(f"Loaded:")
print(f"  Trades: {len(data['trades']):,}")
print(f"  Executors: {len(data['executors']):,}")
print(f"  Controllers: {len(data['controllers']):,}")

# Create trade-to-controller mapping
print("\nCreating trade-to-controller mapping...")
order_to_controller = {}

if 'executors' in data and len(data['executors']) > 0:
    executors_df = data['executors']
    executors_filtered = executors_df[executors_df['net_pnl_quote'] != 0]
    
    for _, executor in executors_filtered.iterrows():
        controller_id = executor.get('controller_id')
        if controller_id and 'custom_info_parsed' in executor and isinstance(executor['custom_info_parsed'], dict):
            order_ids = executor['custom_info_parsed'].get('order_ids', [])
            
            if isinstance(order_ids, np.ndarray):
                order_ids = order_ids.tolist()
            elif not isinstance(order_ids, (list, tuple)):
                order_ids = list(order_ids) if order_ids else []
            
            for oid in order_ids:
                if oid:
                    order_to_controller[str(oid)] = controller_id

# Add controller_id to trades
trades = data['trades'].copy()
trades['controller_id'] = trades['order_id'].map(order_to_controller)
trades['quote_volume'] = trades['amount'] * trades['price']

coverage = (trades['controller_id'].notna().sum() / len(trades)) * 100
print(f"Mapped {len(order_to_controller)} orders to controllers")
print(f"Coverage: {coverage:.1f}% of trades mapped")

print("\n" + "="*80)

INFO:research_notebooks.brigado_v2.data_consolidator:Loaded 11,960 trades from consolidated_trades.parquet
INFO:research_notebooks.brigado_v2.data_consolidator:Loaded 14,401 orders from consolidated_orders.parquet



LOADING CONSOLIDATED DATA



INFO:research_notebooks.brigado_v2.data_consolidator:Loaded 58,227 executors from consolidated_executors.parquet
INFO:research_notebooks.brigado_v2.data_consolidator:Loaded 10 controllers from consolidated_controllers.parquet


Loaded:
  Trades: 11,960
  Executors: 58,227
  Controllers: 10

Creating trade-to-controller mapping...
Mapped 9598 orders to controllers
Coverage: 92.7% of trades mapped



In [3]:
# ============================================================================
#                         FETCH MARKET DATA
# ============================================================================

print("\n" + "="*80)
print("FETCHING MARKET DATA FROM BINANCE")
print("="*80 + "\n")

# Initialize CLOBDataSource
clob = CLOBDataSource()

# Get unique trading pairs and date range
trading_pairs = trades['symbol'].unique()
start_date = trades['timestamp'].min()
end_date = trades['timestamp'].max()
date_range_days = (end_date - start_date).days + 1

print(f"Trading pairs: {', '.join(trading_pairs)}")
print(f"Date range: {start_date.date()} to {end_date.date()} ({date_range_days} days)")
print()

# Fetch market data for each trading pair
market_data = {}

async def fetch_market_data():
    for pair in trading_pairs:
        try:
            # Binance uses no separator (BTC-BRL -> BTCBRL)
            binance_pair = pair.replace('-', '')
            
            print(f"Fetching market data for {pair} ({binance_pair})...")
            
            # Fetch 1-minute candles for the entire date range for accurate market share
            candles = await clob.get_candles(
                connector_name='binance',
                trading_pair=binance_pair,
                interval='1m',
                start_time=int(start_date.timestamp()),
                end_time=int(end_date.timestamp())
            )
            
            if candles.data is not None and len(candles.data) > 0:
                df = candles.data
                
                market_data[pair] = {
                    'open': float(df['open'].iloc[0]),
                    'high': float(df['high'].max()),
                    'low': float(df['low'].min()),
                    'close': float(df['close'].iloc[-1]),
                    'base_volume': float(df['volume'].sum()),
                    'quote_volume': float(df['quote_asset_volume'].sum()) if 'quote_asset_volume' in df.columns else 0,
                    'n_trades': int(df['n_trades'].sum()) if 'n_trades' in df.columns else 0,
                    'high_low_pct': ((float(df['high'].max()) - float(df['low'].min())) / float(df['low'].min()) * 100),
                    'candles_df': df
                }
                
                print(f"  ✓ Market data fetched")
                print(f"    Open: {market_data[pair]['open']:.2f}")
                print(f"    High: {market_data[pair]['high']:.2f}")
                print(f"    Low: {market_data[pair]['low']:.2f}")
                print(f"    Close: {market_data[pair]['close']:.2f}")
                print(f"    Base Volume: {market_data[pair]['base_volume']:,.2f}")
                print(f"    Quote Volume: {market_data[pair]['quote_volume']:,.0f}")
                print(f"    Trades: {market_data[pair]['n_trades']:,}")
                print(f"    High-Low %: {market_data[pair]['high_low_pct']:.2f}%")
            else:
                print(f"  ⚠️  No market data available")
                market_data[pair] = None
                
        except Exception as e:
            print(f"  ✗ Error: {e}")
            import traceback
            traceback.print_exc()
            market_data[pair] = None
        
        print()

# Run async function
await fetch_market_data()

print("="*80)
print(f"Market data fetched for {len([m for m in market_data.values() if m is not None])} / {len(trading_pairs)} pairs")
print("="*80)

INFO:core.data_sources.clob:Initializing ClobDataSource
INFO:MarketFeedsManager:Found connector base: BinancePerpetualBase in core.data_sources.market_feeds.binance_perpetual.binance_perpetual_base
INFO:MarketFeedsManager:Found trades_feed for binance: BinancePerpetualTradesFeed
INFO:MarketFeedsManager:Found oi_feed for binance: BinancePerpetualOIFeed



FETCHING MARKET DATA FROM BINANCE



INFO:hummingbot.connector.exchange.xrpl.xrpl_utils.XRPLNodePool:Initialized XRPLNodePool with 3 nodes, rate limit: 0.3 req/10s, burst tokens: 25/30
INFO:core.data_sources.clob:Fetching data for binance USDTBRL 1m from 1772500876 to 1772670791


Trading pairs: USDT-BRL, BTC-BRL
Date range: 2026-03-03 to 2026-03-05 (2 days)

Fetching market data for USDT-BRL (USDTBRL)...


INFO:core.data_sources.clob:Fetching data for binance BTCBRL 1m from 1772500876 to 1772670791


  ✓ Market data fetched
    Open: 5.18
    High: 5.33
    Low: 5.18
    Close: 5.24
    Base Volume: 127,459,668.20
    Quote Volume: 669,213,490
    Trades: 113,465
    High-Low %: 2.99%

Fetching market data for BTC-BRL (BTCBRL)...
  ✓ Market data fetched
    Open: 357109.00
    High: 386610.00
    Low: 344166.00
    Close: 381046.00
    Base Volume: 405.81
    Quote Volume: 148,878,115
    Trades: 104,775
    High-Low %: 12.33%

Market data fetched for 2 / 2 pairs


In [4]:
# ============================================================================
#                    CALCULATE MARKET SHARE BY TRADING PAIR
# ============================================================================

print("\n" + "="*80)
print("CALCULATING CONTROLLER MARKET SHARE")
print("="*80 + "\n")

market_share_data = {}

for pair in trading_pairs:
    print(f"Processing {pair}:")
    
    pair_trades = trades[trades['symbol'] == pair]
    
    # Calculate controller volumes
    controller_volumes = {}
    for controller_id in pair_trades['controller_id'].dropna().unique():
        ctrl_trades = pair_trades[pair_trades['controller_id'] == controller_id]
        bot_name = ctrl_trades['source_bot'].iloc[0] if len(ctrl_trades) > 0 else 'Unknown'
        
        controller_volumes[controller_id] = {
            'bot_name': bot_name,
            'trades': len(ctrl_trades),
            'base_volume': float(ctrl_trades['amount'].sum()),
            'quote_volume': float(ctrl_trades['quote_volume'].sum())
        }
    
    # Total bot volume for this pair
    total_bot_base_volume = float(pair_trades['amount'].sum())
    total_bot_quote_volume = float(pair_trades['quote_volume'].sum())
    
    # Market share calculations
    if market_data.get(pair) and market_data[pair]:
        market_base_volume = market_data[pair]['base_volume']
        market_quote_volume = market_data[pair]['quote_volume']
        
        if market_base_volume > 0:
            overall_base_share = (total_bot_base_volume / market_base_volume) * 100
        else:
            overall_base_share = 0
            
        if market_quote_volume > 0:
            overall_quote_share = (total_bot_quote_volume / market_quote_volume) * 100
        else:
            overall_quote_share = 0
    else:
        market_base_volume = 0
        market_quote_volume = 0
        overall_base_share = 0
        overall_quote_share = 0
    
    print(f"  Bot base volume: {total_bot_base_volume:,.4f}")
    print(f"  Bot quote volume: {total_bot_quote_volume:,.0f}")
    print(f"  Market base volume: {market_base_volume:,.2f}")
    print(f"  Market quote volume: {market_quote_volume:,.0f}")
    print(f"  Overall market share (base): {overall_base_share:.4f}%")
    print(f"  Overall market share (quote): {overall_quote_share:.4f}%")
    
    # Calculate per-controller market share
    for controller_id, vol_data in controller_volumes.items():
        if market_base_volume > 0:
            controller_base_share = (vol_data['base_volume'] / market_base_volume) * 100
        else:
            controller_base_share = 0
            
        if market_quote_volume > 0:
            controller_quote_share = (vol_data['quote_volume'] / market_quote_volume) * 100
        else:
            controller_quote_share = 0
            
        controller_volumes[controller_id]['base_market_share'] = controller_base_share
        controller_volumes[controller_id]['quote_market_share'] = controller_quote_share
        
        print(f"    {controller_id}: {controller_quote_share:.4f}% (quote)")
    
    market_share_data[pair] = {
        'market_data': market_data.get(pair),
        'total_bot_base_volume': total_bot_base_volume,
        'total_bot_quote_volume': total_bot_quote_volume,
        'total_bot_trades': len(pair_trades),
        'overall_base_market_share': overall_base_share,
        'overall_quote_market_share': overall_quote_share,
        'controllers': controller_volumes
    }
    
    print()

print("="*80)
print("Market share calculation complete!")
print("="*80)


CALCULATING CONTROLLER MARKET SHARE

Processing USDT-BRL:
  Bot base volume: 263,144.3000
  Bot quote volume: 1,381,728
  Market base volume: 127,459,668.20
  Market quote volume: 669,213,490
  Overall market share (base): 0.2065%
  Overall market share (quote): 0.2065%
    brigado-binance-18-1: 0.0797% (quote)
    brigado-binance-21-3: 0.0266% (quote)
    brigado-binance-21-4: 0.0155% (quote)
    brigado-binance-21-1: 0.0259% (quote)
    brigado-binance-21-2: 0.0273% (quote)

Processing BTC-BRL:
  Bot base volume: 7.1577
  Bot quote volume: 2,613,719
  Market base volume: 405.81
  Market quote volume: 148,878,115
  Overall market share (base): 1.7638%
  Overall market share (quote): 1.7556%
    brigado-binance-btcbrl-3: 0.7123% (quote)
    brigado-binance-btcbrl-1: 0.2264% (quote)
    brigado-binance-btcbrl-5: 0.1975% (quote)
    brigado-binance-btcbrl-4: 0.3399% (quote)
    brigado-binance-btcbrl-2: 0.1665% (quote)

Market share calculation complete!


In [5]:
# ============================================================================
#              GENERATE HTML REPORT - MARKET PERFORMANCE
# ============================================================================

print("\n" + "="*80)
print("GENERATING MARKET PERFORMANCE HTML REPORT")
print("="*80 + "\n")

html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Market Performance Analysis - {datetime.now().strftime('%Y-%m-%d %H:%M')}</title>
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}
        
        body {{
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif;
            line-height: 1.6;
            color: #333;
            background: #f5f5f5;
            padding: 20px;
        }}
        
        .container {{
            max-width: 1600px;
            margin: 0 auto;
            background: white;
            border-radius: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
            overflow: hidden;
        }}
        
        .header {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 40px;
            text-align: center;
        }}
        
        .header h1 {{
            font-size: 2.5em;
            margin-bottom: 10px;
            font-weight: 600;
        }}
        
        .header p {{
            font-size: 1.1em;
            opacity: 0.9;
        }}
        
        .content {{
            padding: 40px;
        }}
        
        .section {{
            margin-bottom: 50px;
        }}
        
        .pair-section {{
            background: #f8f9fa;
            border-radius: 10px;
            padding: 30px;
            margin-bottom: 30px;
            border-left: 5px solid #667eea;
        }}
        
        .pair-header {{
            font-size: 2em;
            color: #667eea;
            margin-bottom: 20px;
            display: flex;
            align-items: center;
            justify-content: space-between;
        }}
        
        .market-share-badge {{
            background: #10b981;
            color: white;
            padding: 8px 16px;
            border-radius: 20px;
            font-size: 0.5em;
            font-weight: 600;
        }}
        
        .metrics-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
            gap: 15px;
            margin-bottom: 25px;
        }}
        
        .metric-card {{
            background: white;
            padding: 20px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.05);
        }}
        
        .metric-card h4 {{
            font-size: 0.75em;
            color: #666;
            margin-bottom: 8px;
            text-transform: uppercase;
            letter-spacing: 0.5px;
        }}
        
        .metric-card .value {{
            font-size: 1.5em;
            font-weight: bold;
            color: #333;
        }}
        
        .metric-card.highlight {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
        }}
        
        .metric-card.highlight h4 {{
            color: white;
            opacity: 0.9;
        }}
        
        .metric-card.highlight .value {{
            color: white;
        }}
        
        table {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 20px;
            background: white;
            border-radius: 8px;
            overflow: hidden;
        }}
        
        th, td {{
            padding: 12px;
            text-align: left;
            border-bottom: 1px solid #e0e0e0;
        }}
        
        th {{
            background: #667eea;
            color: white;
            font-weight: 600;
            text-transform: uppercase;
            font-size: 0.75em;
            letter-spacing: 0.5px;
        }}
        
        tr:hover {{
            background: #f5f7fa;
        }}
        
        .bot-name {{
            font-size: 0.8em;
            color: #999;
            font-style: italic;
        }}
        
        .share-high {{
            color: #10b981;
            font-weight: 600;
        }}
        
        .share-medium {{
            color: #f59e0b;
            font-weight: 600;
        }}
        
        .share-low {{
            color: #666;
        }}
        
        .footer {{
            background: #f8f9fa;
            padding: 20px;
            text-align: center;
            color: #666;
            font-size: 0.9em;
        }}
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>📊 Market Performance Analysis</h1>
            <p>Controller Performance vs Market Activity by Trading Pair</p>
            <p style="margin-top: 10px; opacity: 0.9;">Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        </div>
        
        <div class="content">
"""

# Add section for each trading pair
for pair in sorted(market_share_data.keys()):
    data_point = market_share_data[pair]
    mkt_data = data_point['market_data']
    
    # Calculate overall market share display class
    if data_point['overall_quote_market_share'] >= 0.1:
        share_class = "share-high"
    elif data_point['overall_quote_market_share'] >= 0.01:
        share_class = "share-medium"
    else:
        share_class = "share-low"
    
    html_content += f"""
            <div class="pair-section">
                <div class="pair-header">
                    <span>{pair}</span>
                    <span class="market-share-badge">
                        Market Share (Base): <span style="color: white;">{data_point['overall_base_market_share']:.4f}%</span>
                    </span>
                </div>
    """
    
    if mkt_data:
        html_content += f"""
                <h3 style="margin-bottom: 15px; color: #555;">Market Data (since first bot trade: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')})</h3>
                <!-- First Row: Volumes & Trades (Blue) -->
                <div class="metrics-grid">
                    <div class="metric-card highlight">
                        <h4>Total Base Vol</h4>
                        <div class="value">{mkt_data['base_volume']:,.2f}</div>
                    </div>
                    <div class="metric-card highlight">
                        <h4>Total Quote Vol</h4>
                        <div class="value">{mkt_data['quote_volume']:,.0f}</div>
                    </div>
                    <div class="metric-card highlight">
                        <h4>Market Trades</h4>
                        <div class="value">{mkt_data['n_trades']:,}</div>
                    </div>
                </div>
                <!-- Second Row: OHLC (White) -->
                <div class="metrics-grid">
                    <div class="metric-card">
                        <h4>Open</h4>
                        <div class="value">{mkt_data['open']:,.2f}</div>
                    </div>
                    <div class="metric-card">
                        <h4>High</h4>
                        <div class="value">{mkt_data['high']:,.2f}</div>
                    </div>
                    <div class="metric-card">
                        <h4>Low</h4>
                        <div class="value">{mkt_data['low']:,.2f}</div>
                    </div>
                    <div class="metric-card">
                        <h4>Close</h4>
                        <div class="value">{mkt_data['close']:,.2f}</div>
                    </div>
                    <div class="metric-card">
                        <h4>High-Low %</h4>
                        <div class="value">{mkt_data['high_low_pct']:.2f}%</div>
                    </div>
                </div>
        """
    else:
        html_content += """
                <p style="color: #999; font-style: italic; margin-bottom: 20px;">⚠️ Market data not available for this pair</p>
        """
    
    # Controllers table
    if data_point['controllers']:
        html_content += f"""
                <h3 style="margin-top: 30px; margin-bottom: 15px; color: #555;">Controller Performance</h3>
                <table>
                    <thead>
                        <tr>
                            <th>Controller ID</th>
                            <th>Bot Name</th>
                            <th>Trades</th>
                            <th>Base Volume</th>
                            <th>Quote Volume</th>
                            <th>Market Share (Base)</th>
                        </tr>
                    </thead>
                    <tbody>
        """
        
        # Sort controllers by quote volume
        sorted_controllers = sorted(data_point['controllers'].items(), 
                                    key=lambda x: x[1]['quote_volume'], 
                                    reverse=True)
        
        for controller_id, ctrl_data in sorted_controllers:
            # Determine share class
            if ctrl_data['quote_market_share'] >= 0.1:
                share_class = "share-high"
            elif ctrl_data['quote_market_share'] >= 0.01:
                share_class = "share-medium"
            else:
                share_class = "share-low"
            
            html_content += f"""
                        <tr>
                            <td><strong>{controller_id}</strong></td>
                            <td><span class="bot-name">{ctrl_data['bot_name']}</span></td>
                            <td>{ctrl_data['trades']:,}</td>
                            <td>{ctrl_data['base_volume']:,.4f}</td>
                            <td>{ctrl_data['quote_volume']:,.0f}</td>
                            <td class="{share_class}">{ctrl_data['base_market_share']:.4f}%</td>
                        </tr>
            """
        
        html_content += """
                    </tbody>
                </table>
        """
    
    html_content += """
            </div>
    """

# Close HTML
html_content += """
        </div>
        
        <div class="footer">
            <p>Generated by Brigado v2 Market Analysis</p>
            <p style="margin-top: 5px; font-size: 0.85em;">Market data sourced from Binance via CLOBDataSource</p>
        </div>
    </div>
</body>
</html>
"""

# Save HTML report (single file without timestamp)
html_path = file_manager.data_sources_dir / "market_analysis_report.html"
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"✅ Market analysis HTML report generated!")
print(f"\nReport saved to:")
print(f"  - {html_path}")
print(f"\n💡 Open the report in your browser to view market performance analysis.")
print(f"\n📌 Features:")
print(f"  - Grouped by trading pair with market context (OHLC, volume, trades)")
print(f"  - Controller market share calculations (base & quote)")
print(f"  - Bot names shown for reference")
print(f"  - Color-coded market share (green: ≥0.1%, orange: ≥0.01%, gray: <0.01%)")
print("="*80)


GENERATING MARKET PERFORMANCE HTML REPORT

✅ Market analysis HTML report generated!

Report saved to:
  - /Users/tomasgaudino/PycharmProjects/quants-lab/research_notebooks/brigado_v2/data/data_sources/market_analysis_report.html

💡 Open the report in your browser to view market performance analysis.

📌 Features:
  - Grouped by trading pair with market context (OHLC, volume, trades)
  - Controller market share calculations (base & quote)
  - Bot names shown for reference
  - Color-coded market share (green: ≥0.1%, orange: ≥0.01%, gray: <0.01%)
